## Notebook B — Live Data Visualization

Welcome to Notebook B! Here, you will capture the data streamed through OCS and explore it through real-time visualizations.

---
## 6. Receiving OSC in Your App

Below are drop-in snippets for the most common creative coding environments.

### Python

In [1]:
from pythonosc import dispatcher, osc_server
import threading

# Use the same global name as the other cell
global osc_srv 

# --- Cleanup logic (same as visualizer) ---
_prev_srv = globals().get("osc_srv")
if _prev_srv is not None:
    try:
        _prev_srv.shutdown()
        _prev_srv.server_close()
        print("Closed previous server.")
    except: pass

class ReusableOSCUDPServer(osc_server.ThreadingOSCUDPServer):
    allow_reuse_address = True

def on_emg(address, *args):
    rms = (sum(v**2 for v in args) / len(args)) ** 0.5
    print(f"EMG  {len(args):4d} samples  RMS={rms:.4f} mV")

def on_imu_accel(address, *args):
    print(f"Accel   ax={args[0]:+.3f}  ay={args[1]:+.3f}  az={args[2]:+.3f} g")

disp = dispatcher.Dispatcher()
disp.map("/sifi/emg", on_emg)
disp.map("/sifi/imu/accel", on_imu_accel)

osc_srv = ReusableOSCUDPServer(("0.0.0.0", 9000), disp)
print("Listening on port 9000 — press Interrupt (Stop) to stop")

try:
    osc_srv.serve_forever()
except KeyboardInterrupt:
    print("Interrupted by user")
finally:
    osc_srv.shutdown()
    osc_srv.server_close()
    print("Server strictly closed.")

Listening on port 9000 — press Interrupt (Stop) to stop
EMG    61 samples  RMS=0.0000 mV
Accel   ax=+0.491  ay=+7.342  az=-6.594 g
Accel   ax=+0.471  ay=+7.391  az=-6.571 g
Accel   ax=+0.460  ay=+7.368  az=-6.554 g
EMG    61 samples  RMS=0.0000 mV
Accel   ax=+0.463  ay=+7.381  az=-6.539 g
Accel   ax=+0.456  ay=+7.406  az=-6.579 g
Accel   ax=+0.502  ay=+7.399  az=-6.533 g
EMG    61 samples  RMS=0.0000 mV
Accel   ax=+0.565  ay=+7.223  az=-6.527 g
Accel   ax=+0.489  ay=+7.180  az=-6.562 g
Accel   ax=+0.473  ay=+7.269  az=-6.605 g
EMG    59 samples  RMS=0.0000 mV
Accel   ax=+0.473  ay=+7.312  az=-6.601 g
Accel   ax=+0.452  ay=+7.382  az=-6.657 g
Accel   ax=+0.475  ay=+7.508  az=-6.582 g
EMG    61 samples  RMS=0.0000 mV
Accel   ax=+0.569  ay=+7.436  az=-6.482 g
Accel   ax=+0.586  ay=+7.326  az=-6.519 g
Accel   ax=+0.520  ay=+7.345  az=-6.619 g
EMG    61 samples  RMS=0.0000 mV
Accel   ax=+0.468  ay=+7.409  az=-6.661 g
Accel   ax=+0.521  ay=+7.300  az=-6.651 g
Accel   ax=+0.549  ay=+7.282  az

### Max/MSP

```
[udpreceive 9000]
        |
   [oscroute /sifi/emg /sifi/imu/accel /sifi/imu/quat]
     |              |              |
  [unpack]      [unpack]       [unpack]
```

### TouchDesigner

1. Add an **OSC In CHOP** node
2. Set **Network Port** to `9000`
3. Channels `/sifi/emg`, `/sifi/imu/accel`, etc. appear automatically as CHOP channels

### Pure Data

```pd
[netreceive -u -b 9000]
         |
    [oscparse]
         |
    [route /sifi/emg /sifi/imu/accel]
```

### SuperCollider

```supercollider
OSCdef(\emg, { |msg|
    var samples = msg[1..];
    var rms = (samples.squared.mean).sqrt;
    rms.postln;
}, '/sifi/emg');

OSCdef(\accel, { |msg|
    var ax = msg[1], ay = msg[2], az = msg[3];
    [ax, ay, az].postln;
}, '/sifi/imu/accel');
```

### Sending to a different machine

If your creative software runs on a different computer on the same WiFi network, replace `127.0.0.1` in the streamer with that machine's **local IP address** (e.g. `192.168.1.42`). The OSC port must match on both ends.

---
## 7. Verify OSC: Live Plot from OSC Stream

This cell opens an **OSC receiver** on port 9000 and plots what it receives in real time — directly from the OSC messages, not from the device. Use this to confirm that your streamer is working end-to-end before connecting your creative software.

**Run the OSC streamer from section 5 first** (in a separate terminal or a second Jupyter kernel), then run this cell.

In [ ]:
%matplotlib widget

import collections
import threading
import time
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from pythonosc import dispatcher, osc_server

# =============================================================================
# CONFIGURATION
# =============================================================================

LISTEN_PORT = 9000   # Must match OSC_PORT in the streamer (notebook A)

# Rolling buffer sizes — how many samples to display at once.
# At 2000 Hz, 2000 samples = 1 second of EMG history.
# At  100 Hz,  200 samples = 2 seconds of IMU history.
EMG_WINDOW = 10000   # BioPoint single-channel EMG
IMU_WINDOW = 500


# =============================================================================
# CLEANUP -- stop any server left over from a previous run of this cell
#
# Jupyter keeps the kernel alive between executions, so the previous OSC server
# still holds the port. We shut it down and join its thread before rebinding.
# osc_srv and srv_thread are stored as globals so this block can find them.
# =============================================================================

_prev_srv    = globals().get("osc_srv")
_prev_thread = globals().get("srv_thread")

if _prev_srv is not None:
    try:
        _prev_srv.shutdown()      # signals serve_forever() to exit
        _prev_srv.server_close()  # closes the UDP socket
    except Exception:
        pass

if _prev_thread is not None:
    _prev_thread.join(timeout=2)  # wait for the thread to fully exit


# =============================================================================
# RATE TRACKER  (same as in notebook A)
# =============================================================================

class RateTracker:
    def __init__(self, window=2.0):
        self.window  = window
        self._events = collections.deque()

    def add(self, n=1):
        now = time.time()
        self._events.append((now, n))
        cutoff = now - self.window
        while self._events and self._events[0][0] < cutoff:
            self._events.popleft()

    def rate(self):
        if len(self._events) < 2:
            return 0.0
        now     = time.time()
        total   = sum(n for t, n in self._events if t >= now - self.window)
        elapsed = now - self._events[0][0]
        return total / elapsed if elapsed > 0 else 0.0


# =============================================================================
# BUFFERS & RATE TRACKERS
#
# emg_buf         -- BioPoint single-channel EMG (/sifi/emg)
# accel_buf[axis] -- IMU accelerometer           (/sifi/imu/accel)
#
# All deques are filled by the OSC handler thread and read by the plot thread.
# collections.deque is thread-safe for append/popleft from different threads.
# =============================================================================

emg_buf   = collections.deque(maxlen=EMG_WINDOW)
accel_buf = {axis: collections.deque(maxlen=IMU_WINDOW) for axis in ("ax", "ay", "az")}

rt_emg = RateTracker()
rt_imu = RateTracker()


# =============================================================================
# OSC HANDLERS
#
# Called by the OSC server thread whenever a message arrives.
# Keep these fast: just append to the deque and return.
#
#   /sifi/emg       -> burst of N float samples (mV)
#   /sifi/imu/accel -> exactly 3 floats: ax, ay, az (one sample per message)
# =============================================================================

def handle_emg(address, *args):
    emg_buf.extend(args)
    rt_emg.add(len(args))

def handle_accel(address, *args):
    # args is always (ax, ay, az) -- one sample per message
    if len(args) >= 3:
        accel_buf["ax"].append(args[0])
        accel_buf["ay"].append(args[1])
        accel_buf["az"].append(args[2])
        rt_imu.add(1)


# =============================================================================
# START OSC RECEIVER
# =============================================================================

disp = dispatcher.Dispatcher()
disp.map("/sifi/emg",       handle_emg)
disp.map("/sifi/imu/accel", handle_accel)

class ReusableOSCUDPServer(osc_server.ThreadingOSCUDPServer):
    allow_reuse_address = True

try:
    osc_srv = ReusableOSCUDPServer(("0.0.0.0", LISTEN_PORT), disp)
except OSError as e:
    if getattr(e, "winerror", None) == 10048:
        raise RuntimeError(
            f"Port {LISTEN_PORT} is already in use. Stop the other OSC receiver "
            "or change LISTEN_PORT before rerunning."
        ) from e
    raise

srv_thread = threading.Thread(target=osc_srv.serve_forever, daemon=True)
srv_thread.start()

print(f"OSC receiver listening on port {LISTEN_PORT} ...")
print("Make sure the streamer in notebook A is running, then data will appear below.")
time.sleep(1)  # brief pause so the first packets can arrive before the plot opens


# =============================================================================
# LIVE PLOT
#
# Two subplots:
#   Top    -- EMG (single channel)
#   Bottom -- IMU accelerometer (ax / ay / az)
#
# animate() is called every 50 ms (~20 fps). It reads the deques and redraws.
#
# The title shows measured vs nominal sample rates. Compare with notebook A:
#   - notebook A rate low  => loss is in the BLE path (device to computer)
#   - notebook A fine but notebook B low  => loss in the OSC/UDP path
# =============================================================================

fig, (ax_emg, ax_acc) = plt.subplots(2, 1, figsize=(11, 6))

# -- EMG subplot ---------------------------------------------------------------
ax_emg.set_ylabel("mV")
ax_emg.set_xlabel("samples")
ax_emg.set_ylim(-0.001, 0.001)

line_emg, = ax_emg.plot([], [], lw=0.8, color="steelblue", label="EMG")
ax_emg.legend(loc="upper right", fontsize=7)

# -- Accelerometer subplot -----------------------------------------------------
ax_acc.set_title("Accelerometer  (/sifi/imu/accel)")
ax_acc.set_ylabel("g")
ax_acc.set_xlabel("samples")
ax_acc.set_ylim(-30, 30)

line_ax, = ax_acc.plot([], [], lw=1.0, color="tomato",    label="ax")
line_ay, = ax_acc.plot([], [], lw=1.0, color="seagreen",  label="ay")
line_az, = ax_acc.plot([], [], lw=1.0, color="goldenrod", label="az")
ax_acc.legend(loc="upper right")

all_lines = [line_emg, line_ax, line_ay, line_az]


def animate(_):
    # -- BioPoint EMG ----------------------------------------------------------
    y = list(emg_buf)
    if y:
        line_emg.set_data(range(len(y)), y)
        ax_emg.set_xlim(0, max(len(y), 1))
        ax_emg.set_title("EMG  (/sifi/emg)")
    else:
        line_emg.set_data([], [])

    # -- Accelerometer ---------------------------------------------------------
    n_acc = len(accel_buf["ax"])
    for line, key in zip([line_ax, line_ay, line_az], ["ax", "ay", "az"]):
        vals = list(accel_buf[key])
        line.set_data(range(len(vals)), vals) if vals else line.set_data([], [])
    if n_acc:
        ax_acc.set_xlim(0, n_acc)

    # -- Title with live rate measurements -------------------------------------
    emg_rate = rt_emg.rate()
    imu_rate = rt_imu.rate()

    parts = []
    if emg_rate > 0:
        parts.append(f"EMG: {emg_rate:.0f} Hz (nom 2000)")
    parts.append(f"IMU: {imu_rate:.0f} Hz (nom 100)")

    fig.suptitle("BioPoint via OSC -- Live Verification  |  " + "   ".join(parts), fontsize=10)

    return all_lines


# blit=False required because suptitle and ax titles live outside the axes
ani = animation.FuncAnimation(fig, animate, interval=50, blit=False, cache_frame_data=False)
plt.tight_layout()
plt.show()

# To stop the OSC receiver, run the cell below.


In [ ]:
# Run this cell to stop the OSC receiver cleanly.
osc_srv.shutdown()     # waits for serve_forever() to exit
osc_srv.server_close() # releases the UDP socket
srv_thread.join(timeout=2)
print("OSC receiver stopped.")

---
## 8. OSC Address Reference

| OSC Address | Value types | Signal | Nominal rate |
|---|---|---|---|
| `/sifi/emg/sample_rate` | `float` | — | sent once at start |
| `/sifi/imu/sample_rate` | `float` | — | sent once at start |
| `/sifi/emg` | `float, float, …` | EMG burst (mV) | 2000 Hz |
| `/sifi/ecg` | `float, float, …` | ECG burst (mV) | 500 Hz |
| `/sifi/eda` | `float, float, …` | EDA burst (µS) | 50 Hz |
| `/sifi/ppg/ir` | `float, float, …` | PPG infrared | 100 Hz |
| `/sifi/ppg/r` | `float, float, …` | PPG red | 100 Hz |
| `/sifi/ppg/g` | `float, float, …` | PPG green | 100 Hz |
| `/sifi/ppg/b` | `float, float, …` | PPG blue | 100 Hz |
| `/sifi/imu/accel` | `ax, ay, az` | Acceleration (g) | 100 Hz |
| `/sifi/imu/quat` | `qw, qx, qy, qz` | Orientation quaternion | 100 Hz |
| `/sifi/temperature` | `float` | Temperature (°C) | 1 Hz |

---

## Tips & Troubleshooting

**Device not found during scan?**
- Make sure it is charged and switched on — look for a LED blinking
- On Windows, you may need to pair the device in Bluetooth Settings first
- Keep the device within 2 m of your laptop during the initial scan

**OSC messages not arriving in my app?**
- Check that the port numbers match on both the sender and receiver
- For cross-machine streaming, replace `127.0.0.1` with the receiver's LAN IP address
- Firewalls can block UDP — temporarily disable or add an inbound exception for the port

**Signal looks very noisy?**
- For EMG/ECG: ensure good skin contact — moisten the electrode pads slightly
- For EDA: the sensor must sit flat against skin, not over hair
- Verify `set_filters(enable=True)` was called before `start()`


Have fun building at MishMash 2026!  
Questions? Find the SiFi Labs team at the hardware table. More documentation can be found here: [docs.sifilabs.com/](https://docs.sifilabs.com/).